# v9f: PATCHING + STEERING: does the self carry WHICH LANGUAGE is written? (GPU 1, no training)
- Models: mixed 45 (knows all 3 languages best), distant 30, related 15. Decoder layers only (the encoder only sees English).
- Groups: 30% of each decoder layer: self / look-alike task / random task (3 draws).
- PATCH: model asked for language A; the group's neurons are replaced by their activity when the SAME sentence is translated into B.
- STEER: add alpha x (average activity in B - in A) to the group's neurons (alpha 1, 2, 4).
- Measured: does the B translation become more likely (shift to B)? does A get worse? and in generation: % of output in B's script.
- Prediction if the self is shared machinery: patching the self barely moves the output to B; task neurons move it more.
Results: `analysis/v9/patch/`. Progress: `tail -f logs/patch_v9.log`.

In [ ]:
import os, sys, glob, json, time, subprocess
os.environ["HF_HUB_DISABLE_XET"] = "1"
GPU = "1"
REPO = "/workspace/shreshth"; os.chdir(REPO); sys.path.insert(0, REPO)
from IPython.display import Image, display
from huggingface_hub import HfApi, hf_hub_download
for attempt in range(5):
    try:
        api = HfApi(); HF_REPO = api.whoami()["name"] + "/cml-lang-self-checkpoints"; FILES = api.list_repo_files(HF_REPO); break
    except Exception as e:
        print("HF retry", attempt, type(e).__name__); time.sleep(10)

def get(run, phases):
    for ph in phases:
        if glob.glob(f"runs/{run}/ckpt/p{ph:03d}_*.pt"):
            continue
        name = [f for f in FILES if f.startswith(f"{run}/ckpt/p{ph:03d}_")]
        if name:
            hf_hub_download(HF_REPO, name[0], local_dir="runs")
        else:
            print("missing (not here, not on HF):", run, ph)

def last_phase(run):
    phs = [int(os.path.basename(f)[1:4]) for f in FILES if f.startswith(f"{run}/ckpt/p")]
    phs += [int(os.path.basename(f)[1:4]) for f in glob.glob(f"runs/{run}/ckpt/p*.pt")]
    return max(phs)

def run(langset, *cmd):
    """runs one call, shows the output here AND appends it to logs/patch_v9.log (watch with: tail -f logs/patch_v9.log)"""
    env = {**os.environ, "CML_LANGSET": langset}; os.makedirs("logs", exist_ok=True)
    p = subprocess.Popen([sys.executable, "-u", "patch_v9.py", "--gpu", GPU, *cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    with open("logs/patch_v9.log", "a") as log:
        for line in p.stdout:
            print(line, end=""); log.write(line); log.flush()
    p.wait(); print("exit code", p.returncode)

DIST, REL, MIX = "v4b_continual_s0", "v8_indic_continual_s0", "v6b_mixed_s0"
STARTS = [("distant", "mixed45", MIX, 45), ("distant", "distant30", DIST, 30), ("indic", "related15", REL, 15)]
for _, _, r, ph in STARTS:
    get(r, [ph])
print("start models:", [(n, r, ph) for _, n, r, ph in STARTS])

In [ ]:
need = [f"data/en-{l}/train.jsonl" for l in ["hi", "zh", "gu"]] + [f"data/flores/devtest.{l}.jsonl" for l in ["en", "hi", "zh", "gu"]]
need += [f"data/flores/dev.{l}.jsonl" for l in ["en", "hi", "zh", "ar", "gu", "pa"]] + [f"data/en-{l}/val.jsonl" for l in ["hi", "zh", "ar", "gu", "pa"]]
miss = [f for f in need if not os.path.exists(f)]
print("all there" if not miss else f"MISSING: {miss}  -> run your download snippet first, then re-run this cell")

In [ ]:
for langset, name, r, ph in STARTS:
    run(langset, "--start", f"{name}={r}:{ph}")
subprocess.run([sys.executable, "patch_v9.py", "--summarize"])
display(Image("analysis/v9/patch/fig1_shift_to_B.png"))